<a href="https://colab.research.google.com/github/AM0ya/Clase_Programacion/blob/main/Sesion12_Evaluacion_Datos_Categoricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**:Alfredo Moya Orozco
- **Matrícula**: 269712

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [4]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [7]:
# 1.1 — Recorre todas las columnas categóricas con .value_counts() o .unique()
# para inspeccionar sus valores.
columnas_categoricas=['gender','SeniorCitizen','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Churn']

for columna in columnas_categoricas:
  print(df[columna].value_counts())
  print()



gender
Male      3555
Female    3488
Name: count, dtype: int64

SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64

Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

TechSupport
No                     3473
Yes                    2044
No internet service    15

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta:_ En la revision de las columnas se encontro que la columna Contracts tiene tres valores distintos y existen dos formatos diferentes. Uno de ellos esta separado por guines medios "Month-to-month", y las otras dos por espacios "Two year & One year". En si no es un error, pero por cuestion de formato podria preferirse normalizar la informacion. En caso de querer cambiar la que esta separada por guines medios se utilizaria el siguiente codigo:

df['Contract'] = df['Contract'].str.replace('-', ' ')

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [8]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [9]:
# 2.1 — Verifica: ¿las filas con 'No internet service' en OnlineSecurity
# coinciden con las filas donde InternetService == 'No'?
# (pista: cruza ambas columnas con pd.crosstab o filtrando)

# Creamos una tabla cruzada (crosstab) entre ambas columnas para validar la relación
validacion_internet = pd.crosstab(df['InternetService'], df['OnlineSecurity'])
display(validacion_internet)

OnlineSecurity,No,No internet service,Yes
InternetService,,,
DSL,1241,0,1180
Fiber optic,2257,0,839
No,0,1526,0


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_No internet service no es un valor inválido (como un error tipográfico), sino una categoría legítima y lógica. La verificación en 2.1 demuestra que las 1,526 filas que tienen 'No' en InternetService coinciden exactamente con las 1,526 filas que tienen 'No internet service' en OnlineSecurity (y lo mismo ocurre en los demás servicios).
En lugar de eliminar estos datos, los dejaría tal cual o los agruparía/recodificaría como 'No' (asumiendo que si no tienen internet, tampoco tienen el servicio de seguridad), ya que para un modelo predictivo, mantener la etiqueta redundante 'No internet service' en 6 columnas distintas añade información repetitiva que ya explica la columna InternetService."

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [10]:
# 3.1 — Calcula .nunique() para TODAS las columnas del dataset y la razón (valores únicos / total de filas)
total_filas = len(df)

# Creamos un DataFrame para mostrar ambos resultados de forma clara
cardinalidad_df = pd.DataFrame({
    'Valores Únicos (nunique)': df.nunique(),
    'Razón de Cardinalidad': df.nunique() / total_filas
})

display(cardinalidad_df.sort_values(by='Valores Únicos (nunique)', ascending=False))

,Valores Únicos (nunique),Razón de Cardinalidad
customerID,7043,1.000000
TotalCharges,6531,0.927304
MonthlyCharges,1585,0.225046
tenure,73,0.010365
PaymentMethod,4,0.000568
StreamingMovies,3,0.000426
TechSupport,3,0.000426
OnlineBackup,3,0.000426
StreamingTV,3,0.000426
DeviceProtection,3,0.000426


**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_CustomerID y TotalCharges, aunque esta última es numérica presentan una alta cardinalidad.
Al ser un identificador único, un modelo predictivo (como un árbol de decisión) simplemente se memorizaría qué cliente canceló (Churn) usando su ID en lugar de aprender patrones reales. Esto produce un sobreajuste masivo (overfitting) y hace que el modelo sea completamente inútil con datos nuevos, ya que los nuevos clientes tendrán IDs que el modelo jamás ha visto.


**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [15]:
# Identificamos los 10 IDs de cliente más frecuentes
top_10_ids = df['customerID'].value_counts().head(10).index

# Aplicamos la técnica: si el ID está en el Top 10 se queda igual, si no, se reemplaza por 'Otros'
df['customerID_grouped'] = df['customerID'].where(df['customerID'].isin(top_10_ids), other='Otros')

# Mostramos las frecuencias de la nueva columna agrupada
display(df['customerID_grouped'].value_counts())

,count
customerID_grouped,
Otros,7033
7590-VHVEG,1
5575-GNVDE,1
9837-FWLCH,1
1699-HPSBG,1
7203-OYKCT,1
1035-IPQPU,1
7398-LXGYX,1
2823-LKABH,1


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_No, no es útil. Al agrupar casi la totalidad de tus datos (7,033 de 7,043 registros) en una única categoría llamada 'Otros', has destruido prácticamente toda la variabilidad de la columna. El modelo solo verá un grupo gigantesco uniforme y no aprenderá nada relevante de él.
En el caso de country: Hay países dominantes muy claros (como Estados Unidos, India, Reino Unido) donde se concentra una gran proporción del contenido de Netflix, por lo que agrupar los países minoritarios en 'Otros' mantiene categorías representativas.
En el caso de customerID: Al ser un identificador único donde cada cliente aparece exactamente una vez, no existen "categorías dominantes". Aplicar el Top 10 solo selecciona 10 clientes al azar y amontona al 99.9% de los clientes en 'Otros'. El problema real de los identificadores únicos no se puede solucionar agrupando; la única solución real es eliminar la columna por completo antes de entrenar el modelo predictivo.

---
## Actividad 4 — Tipos de dato (30 pts)

In [17]:
df['TotalCharges'].dtype
# Busquemos los registros donde 'TotalCharges' tiene espacios en blanco o valores no numéricos
valores_problematicos = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]

print(f"Cantidad de filas problemáticas: {len(valores_problematicos)}")
print("\nEjemplo de las filas con problemas:")
display(valores_problematicos[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

Cantidad de filas problemáticas: 11

Ejemplo de las filas con problemas:


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,


```markdown
**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_
Tras realizar la inspección, se detectó que existen **11 filas**
donde la columna TotalCharges contiene un **espacio en blanco (' ')** en lugar de un número.

Al analizar estos registros, se observa que todos corresponden a clientes con
tenure = 0 (antigüedad de cero meses). Esto significa que son clientes
completamente nuevos que aún no han completado su primer ciclo de
facturación. Como no se ha generado ningún cobro, el dataset original
registró su cargo total como un texto vacío, lo cual forzó a pandas a
interpretar toda la columna como tipo de dato `object` (texto).

In [21]:
# 4.2 — Corrige el tipo de TotalCharges.
# Pista: pd.to_numeric() con el parámetro errors= te puede ayudar a identificar
# o manejar los valores problemáticos que encontraste en 4.1.


# Guardamos los cambios convirtiendo la columna a numérico de forma permanente.
# Los espacios en blanco se convertirán en NaN (valores nulos).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verificamos que el tipo de dato ahora sea float64
print("Nuevo tipo de dato para TotalCharges:", df['TotalCharges'].dtype)
print(f"Cantidad de valores nulos (NaN) generados: {df['TotalCharges'].isna().sum()}")


Nuevo tipo de dato para TotalCharges: float64
Cantidad de valores nulos (NaN) generados: 11


In [23]:
# 4.3 — Convierte a category las columnas categóricas que, según lo que calculaste
# en la Actividad 3, tengan cardinalidad baja y valores fijos.
# Verifica con .dtypes que el cambio se aplicó correctamente.

# Definimos las columnas categóricas que tienen pocos valores únicos posibles
columnas_categoricas_baja = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn'
]

# Realizamos la conversión al tipo de dato 'category'
for col in columnas_categoricas_baja:
    df[col] = df[col].astype('category')

# Verificamos que los tipos de datos se hayan actualizado correctamente
display(df[columnas_categoricas_baja].dtypes)

,0
gender,category
SeniorCitizen,category
Partner,category
Dependents,category
PhoneService,category
MultipleLines,category
InternetService,category
OnlineSecurity,category
OnlineBackup,category
DeviceProtection,category


```markdown
---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?
2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_

**1. Alertas más fáciles y difíciles de decidir:**
* **La más fácil:** Fue la de **tipos de datos e inconsistencias (Actividad 4 y 1)**.
Encontrar que la columna `TotalCharges` era de tipo `object` debido a los 11
 espacios en blanco de clientes con `tenure = 0` fue sumamente directo
 gracias al análisis lógico de los datos. Normalizar la columna con `pd.
 to_numeric` y transformarla a `float64` es una corrección técnica
 obligatoria y estándar.

* **La más difícil:** Fue la de **valores redundantes / "inválidos"
(Actividad 2)**. Determinar qué hacer con `'No internet service'` en
múltiples columnas requiere decisiones de negocio. Aunque verificamos que es
 una categoría lógica y libre de errores tipográficos, decidir si se agrupa
 bajo la etiqueta `'No'` o si se deja intacta implica evaluar si el modelo
 predictivo se beneficiará de la simplificación o si perderá
 interpretabilidad.


**2. Recomendaciones para el modelado de datos:**
* **Sobre `customerID`:** Le advertiría firmemente que **no la utilice bajo
ninguna circunstancia** en el entrenamiento del modelo. Al ser un
identificador único (cardinalidad del 100%), provocará un sobreajuste severo
 (overfitting). También le explicaría que agruparla usando la técnica "Top 10 + Otros"
 es inútil aquí porque no hay categorías dominantes (destruye toda
  la variabilidad de la columna), por lo que la única acción correcta es
  **eliminar la columna por completo**.

* **Sobre `TotalCharges`:** Le informaría que la columna ya fue corregida a
 tipo numérico (`float64`). Sin embargo, le advertiría que preste atención a
  los **11 valores nulos (NaN)** que se generaron. Estos valores corresponden
   a clientes completamente nuevos (`tenure = 0`), por lo que antes de
   entrenar el modelo deberá decidir si imputarles un valor de `0` (lo cual
   es lógico dado que no han facturado) o eliminar esas 11 filas.
```

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.